In [0]:
import hashlib
dbutils.widgets.text("run_id", "")
dbutils.widgets.text("run_open_ts", "")
dbutils.widgets.text("source_update_id", "")
dbutils.widgets.text("silver_update_id", "")
dbutils.widgets.text("scratch_prefix", "")
RUN = dbutils.widgets.get("run_id").strip()
RUN_OPEN_TS = dbutils.widgets.get("run_open_ts").strip()
SOURCE_UPDATE_ID = dbutils.widgets.get("source_update_id").strip()
SILVER_UPDATE_ID = dbutils.widgets.get("silver_update_id").strip()
SCRATCH_PREFIX = dbutils.widgets.get("scratch_prefix").strip()
assert RUN.startswith("dq4_omop_") and RUN.replace("_", "").isalnum(), RUN
assert RUN_OPEN_TS and SOURCE_UPDATE_ID and SILVER_UPDATE_ID and SCRATCH_PREFIX
LANE="omop_issueify"
STATEMENTS=["-- DQ4 OMOP issue-ification pack. Pinned run: dq4_omop_20260825_r5.\n-- OMOP pipeline update: cc682c9c-8795-4c48-adea-f988320f8d0d.\n\nINSERT INTO `8_dev`.silver_qc.dq_check\n  (check_id, family, kahn, engine, target_table, target_column, params,\n   sql_template, rule_source, severity_default, enabled, created_by_session, created_at)\nWITH d AS (\n  SELECT DISTINCT check_id, dqd_checkid, lower(category) AS kahn,\n         lower(check_name) AS check_name, cdm_table, cdm_field,\n         check_level, check_description, threshold, concept_id, unit_concept_id\n  FROM `8_dev`.silver_qc.v_dqd_canonical\n  WHERE run_id = 'dq4_omop_20260825_r5')\nSELECT d.check_id, concat('dqd_', d.kahn), d.kahn, 'dqd',\n       concat('8_dev.omop_cdm.', d.cdm_table), d.cdm_field,\n       map('dqd_checkid', d.dqd_checkid,\n           'checkLevel', d.check_level, 'checkName', d.check_name,\n           'conceptId', coalesce(d.concept_id, ''),\n           'unitConceptId', coalesce(d.unit_concept_id, ''),\n           'thresholdValue', coalesce(CAST(d.threshold AS STRING), ''),\n           'checkDescription', coalesce(d.check_description, '')),\n       NULL, 'dqd_2.8.9', NULL, false, 'DQ4', current_timestamp()\nFROM d\nLEFT JOIN `8_dev`.silver_qc.dq_check k ON k.check_id = d.check_id\nWHERE k.check_id IS NULL","INSERT INTO `8_dev`.silver_qc.dq_check\n  (check_id, family, kahn, engine, target_table, target_column, params,\n   sql_template, rule_source, severity_default, enabled, created_by_session, created_at)\nWITH h AS (\n  SELECT DISTINCT concat('heel:', rule_id) AS check_id, rule_id, rule_name\n  FROM `8_dev`.silver_qc.heel_rule_disposition\n  WHERE run_id = 'dq4_omop_20260825_r5' AND disposition = 'lifted')\nSELECT h.check_id, 'heel_rule', 'plausibility', 'heel', NULL, NULL,\n       map('rule_id', h.rule_id, 'rule_name', h.rule_name,\n           'source_tag', 'achilles v1.6.3'),\n       NULL, 'achilles_heel_1.6.3', NULL, false, 'DQ4', current_timestamp()\nFROM h\nLEFT JOIN `8_dev`.silver_qc.dq_check k ON k.check_id = h.check_id\nWHERE k.check_id IS NULL","MERGE INTO `8_dev`.silver_qc.dq_issue t\nUSING (\n  SELECT sha2(concat_ws('|', check_id, coalesce(target_table, 'None'),\n                        coalesce(target_column, 'None'), '~'), 256) AS issue_id,\n         check_id, target_table, target_column,\n         CAST(NULL AS STRING) AS lane_qualifier, title, prevalence, severity_tier\n  FROM (\n    SELECT check_id,\n           concat('8_dev.omop_cdm.', cdm_table) AS target_table,\n           cdm_field AS target_column,\n           concat(check_name, ' failed on ', cdm_table,\n                  CASE WHEN cdm_field IS NULL THEN '' ELSE concat('.', cdm_field) END,\n                  CASE WHEN concept_id IS NULL THEN ''\n                       ELSE concat(' [concept ', concept_id,\n                                   CASE WHEN unit_concept_id IS NULL THEN ''\n                                        ELSE concat('/unit ', unit_concept_id) END, ']') END,\n                  ' (', CAST(measured_rows AS STRING), ' rows)') AS title,\n           named_struct('measured_rows', measured_rows,\n                        'total_rows', total_rows,\n                        'approximate', false,\n                        'measured_at', CAST('2026-08-25' AS DATE),\n                        'source_update_id', 'cc682c9c-8795-4c48-adea-f988320f8d0d') AS prevalence,\n           CASE\n             WHEN lower(check_name) IN ('cdmtable','cdmfield','isprimarykey','isforeignkey','cdmdatatype') THEN 'P0'\n             WHEN measured_rows >= 1000000 OR pct_violated >= 0.01 THEN 'P1'\n             WHEN measured_rows >= 10000 OR pct_violated >= 0.0001 THEN 'P2'\n             ELSE 'P3' END AS severity_tier\n    FROM `8_dev`.silver_qc.v_dqd_canonical\n    WHERE run_id='dq4_omop_20260825_r5' AND failed=1)) s\nON t.issue_id=s.issue_id\nWHEN MATCHED THEN UPDATE SET\n  t.last_seen_run='dq4_omop_20260825_r5', t.prevalence=s.prevalence,\n  t.severity_tier=s.severity_tier,\n  t.status=CASE\n    WHEN t.status='closed' THEN 'open'\n    WHEN t.status='not_an_issue' AND (\n      coalesce(t.prevalence.measured_rows,-1)<>coalesce(s.prevalence.measured_rows,-1) OR\n      coalesce(t.prevalence.total_rows,-1)<>coalesce(s.prevalence.total_rows,-1) OR\n      coalesce(t.severity_tier,'')<>coalesce(s.severity_tier,'') OR\n      coalesce(t.title,'')<>coalesce(s.title,'')) THEN 'open'\n    ELSE t.status END,\n  t.updated_at=current_timestamp()\nWHEN NOT MATCHED THEN INSERT\n  (issue_id,check_id,target_table,target_column,lane_qualifier,title,\n   first_seen_run,last_seen_run,prevalence,severity_tier,status,\n   dedupe_cluster_id,evidence_hash,g1_rule_id,g1_action,\n   created_by_session,created_at,updated_at)\nVALUES (s.issue_id,s.check_id,s.target_table,s.target_column,s.lane_qualifier,s.title,\n        'dq4_omop_20260825_r5','dq4_omop_20260825_r5',s.prevalence,s.severity_tier,'open',\n        NULL,NULL,NULL,NULL,'DQ4',current_timestamp(),current_timestamp())","MERGE INTO `8_dev`.silver_qc.dq_issue t\nUSING (\n  SELECT sha2(concat_ws('|', check_id, coalesce(target_table, 'None'), 'None', lane_qualifier), 256) AS issue_id,\n         check_id, target_table, CAST(NULL AS STRING) AS target_column,\n         lane_qualifier, title, prevalence, severity_tier\n  FROM (\n    SELECT concat('heel:', h.rule_id) AS check_id,\n           CASE lower(trim(a.category))\n             WHEN 'person' THEN '8_dev.omop_cdm.person'\n             WHEN 'death' THEN '8_dev.omop_cdm.death'\n             WHEN 'location' THEN '8_dev.omop_cdm.location'\n             WHEN 'care site' THEN '8_dev.omop_cdm.care_site'\n             WHEN 'provider' THEN '8_dev.omop_cdm.provider'\n             WHEN 'visit occurrence' THEN '8_dev.omop_cdm.visit_occurrence'\n             WHEN 'visit detail' THEN '8_dev.omop_cdm.visit_detail'\n             WHEN 'observation period' THEN '8_dev.omop_cdm.observation_period'\n             WHEN 'condition occurrence' THEN '8_dev.omop_cdm.condition_occurrence'\n             WHEN 'procedure occurrence' THEN '8_dev.omop_cdm.procedure_occurrence'\n             WHEN 'observation' THEN '8_dev.omop_cdm.observation'\n             WHEN 'device exposure' THEN '8_dev.omop_cdm.device_exposure'\n             WHEN 'specimen' THEN '8_dev.omop_cdm.specimen'\n             WHEN 'measurement' THEN '8_dev.omop_cdm.measurement'\n             ELSE NULL END AS target_table,\n           concat_ws('|', concat('analysis_', coalesce(CAST(h.analysis_id AS STRING), 'na')),\n                     substr(sha2(coalesce(h.achilles_heel_warning, ''), 256), 1, 16)) AS lane_qualifier,\n           concat('Heel rule ', h.rule_id, ': ', coalesce(h.achilles_heel_warning, 'warning')) AS title,\n           named_struct('measured_rows', CAST(coalesce(h.record_count,0) AS BIGINT),\n                        'total_rows', CAST(NULL AS BIGINT),\n                        'approximate', true,\n                        'measured_at', CAST('2026-08-25' AS DATE),\n                        'source_update_id', 'cc682c9c-8795-4c48-adea-f988320f8d0d') AS prevalence,\n           CASE WHEN coalesce(h.record_count,0)>=1000000 THEN 'P1'\n                WHEN coalesce(h.record_count,0)>=10000 THEN 'P2' ELSE 'P3' END AS severity_tier\n    FROM `8_dev`.silver_qc.heel_results h\n    LEFT JOIN (SELECT DISTINCT analysis_id, category\n               FROM `8_dev`.silver_qc.achilles_analysis\n               WHERE run_id='dq4_omop_20260825_r5') a ON a.analysis_id=h.analysis_id\n    WHERE h.run_id='dq4_omop_20260825_r5')) s\nON t.issue_id=s.issue_id\nWHEN MATCHED THEN UPDATE SET\n  t.last_seen_run='dq4_omop_20260825_r5',t.prevalence=s.prevalence,\n  t.severity_tier=s.severity_tier,\n  t.status=CASE\n    WHEN t.status='closed' THEN 'open'\n    WHEN t.status='not_an_issue' AND (\n      coalesce(t.prevalence.measured_rows,-1)<>coalesce(s.prevalence.measured_rows,-1) OR\n      coalesce(t.prevalence.total_rows,-1)<>coalesce(s.prevalence.total_rows,-1) OR\n      coalesce(t.severity_tier,'')<>coalesce(s.severity_tier,'') OR\n      coalesce(t.title,'')<>coalesce(s.title,'')) THEN 'open'\n    ELSE t.status END,\n  t.updated_at=current_timestamp()\nWHEN NOT MATCHED THEN INSERT\n  (issue_id,check_id,target_table,target_column,lane_qualifier,title,\n   first_seen_run,last_seen_run,prevalence,severity_tier,status,\n   dedupe_cluster_id,evidence_hash,g1_rule_id,g1_action,\n   created_by_session,created_at,updated_at)\nVALUES (s.issue_id,s.check_id,s.target_table,s.target_column,s.lane_qualifier,s.title,\n        'dq4_omop_20260825_r5','dq4_omop_20260825_r5',s.prevalence,s.severity_tier,'open',\n        NULL,NULL,NULL,NULL,'DQ4',current_timestamp(),current_timestamp())","MERGE INTO `8_dev`.silver_qc.dq_issue t\nUSING (\n  SELECT sha2(concat_ws('|', r.check_id, coalesce(k.target_table, 'None'),\n                        coalesce(k.target_column, 'None'), '~'), 256) AS issue_id,\n         r.check_id,k.target_table,k.target_column,CAST(NULL AS STRING) AS lane_qualifier,\n         concat(r.check_id,' violated (',CAST(r.measured_rows AS STRING),' of ',CAST(r.total_rows AS STRING),' rows)') AS title,\n         named_struct('measured_rows',r.measured_rows,\n                      'total_rows',r.total_rows,\n                      'approximate',false,\n                      'measured_at',CAST('2026-08-25' AS DATE),\n                      'source_update_id','cc682c9c-8795-4c48-adea-f988320f8d0d') AS prevalence,\n         CASE WHEN k.severity_default='P0' THEN 'P0'\n              WHEN r.measured_rows>=1000000 OR r.measured_rows/nullif(r.total_rows,0)>=0.01 THEN 'P1'\n              WHEN r.measured_rows>=10000 OR r.measured_rows/nullif(r.total_rows,0)>=0.0001 THEN 'P2'\n              ELSE 'P3' END AS severity_tier\n  FROM `8_dev`.silver_qc.dq_check_result r\n  JOIN `8_dev`.silver_qc.dq_check k ON k.check_id=r.check_id\n  WHERE r.run_id='dq4_omop_20260825_r5' AND r.status='fail' AND k.rule_source='themis') s\nON t.issue_id=s.issue_id\nWHEN MATCHED THEN UPDATE SET\n  t.last_seen_run='dq4_omop_20260825_r5',t.prevalence=s.prevalence,\n  t.severity_tier=s.severity_tier,\n  t.status=CASE\n    WHEN t.status='closed' THEN 'open'\n    WHEN t.status='not_an_issue' AND (\n      coalesce(t.prevalence.measured_rows,-1)<>coalesce(s.prevalence.measured_rows,-1) OR\n      coalesce(t.prevalence.total_rows,-1)<>coalesce(s.prevalence.total_rows,-1) OR\n      coalesce(t.severity_tier,'')<>coalesce(s.severity_tier,'') OR\n      coalesce(t.title,'')<>coalesce(s.title,'')) THEN 'open'\n    ELSE t.status END,\n  t.updated_at=current_timestamp()\nWHEN NOT MATCHED THEN INSERT\n  (issue_id,check_id,target_table,target_column,lane_qualifier,title,\n   first_seen_run,last_seen_run,prevalence,severity_tier,status,\n   dedupe_cluster_id,evidence_hash,g1_rule_id,g1_action,\n   created_by_session,created_at,updated_at)\nVALUES (s.issue_id,s.check_id,s.target_table,s.target_column,s.lane_qualifier,s.title,\n        'dq4_omop_20260825_r5','dq4_omop_20260825_r5',s.prevalence,s.severity_tier,'open',\n        NULL,NULL,NULL,NULL,'DQ4',current_timestamp(),current_timestamp())","CREATE OR REPLACE TABLE `8_dev`.silver_qc_tmp.dq4_scope_dq4_omop_20260825_r5 AS\nSELECT DISTINCT check_id FROM `8_dev`.silver_qc.v_dqd_canonical WHERE run_id='dq4_omop_20260825_r5'\nUNION\nSELECT DISTINCT check_id FROM `8_dev`.silver_qc.dq_check_result WHERE run_id='dq4_omop_20260825_r5'\nUNION\nSELECT DISTINCT concat('heel:',rule_id) FROM `8_dev`.silver_qc.heel_rule_disposition\nWHERE run_id='dq4_omop_20260825_r5' AND disposition='lifted'","MERGE INTO `8_dev`.silver_qc.dq_issue t\nUSING `8_dev`.silver_qc_tmp.dq4_scope_dq4_omop_20260825_r5 s\nON t.check_id=s.check_id\nWHEN MATCHED AND t.last_seen_run<>'dq4_omop_20260825_r5'\n  AND t.status IN ('open','triaged','verified','routed')\nTHEN UPDATE SET t.status='closed',t.updated_at=current_timestamp()","CREATE OR REPLACE TABLE `8_dev`.silver_qc_tmp.dq4_bundle_dq4_omop_20260825_r5 AS\nSELECT i.issue_id,\n       to_json(named_struct(\n         'issue_id',i.issue_id,'title',i.title,'target_table',i.target_table,\n         'target_column',i.target_column,'lane_qualifier',i.lane_qualifier,\n         'severity_tier',i.severity_tier,'g1_rule_id',i.g1_rule_id,\n         'g1_action',i.g1_action,'prevalence',i.prevalence)) AS issue_json,\n       to_json(named_struct('check_id',k.check_id,'params',k.params)) AS check_json,\n       to_json(named_struct(\n         'cdm_table',x.cdm_table,'silver_products',x.silver_products,\n         'governed_lookups',x.governed_lookups,'crosswalk_note',x.note,\n         'bronze_routes',sort_array(collect_set(l.source_table)))) AS lineage_json,\n       array('omop_lane') AS context_ids\nFROM `8_dev`.silver_qc.dq_issue i\nJOIN `8_dev`.silver_qc.dq_check k ON k.check_id=i.check_id\nLEFT JOIN `8_dev`.silver_qc.omop_silver_crosswalk x\n  ON x.cdm_table=regexp_extract(i.target_table,'^8_dev\\\\.omop_cdm\\\\.([\\\\s\\\\S]+)$',1)\nLEFT JOIN `8_dev`.silver_qc.product_lineage l\n  ON array_contains(x.silver_products,l.product_key)\nWHERE i.last_seen_run='dq4_omop_20260825_r5'\nGROUP BY i.issue_id,i.title,i.target_table,i.target_column,i.lane_qualifier,\n         i.severity_tier,i.g1_rule_id,i.g1_action,i.prevalence,\n         k.check_id,k.params,x.cdm_table,x.silver_products,x.governed_lookups,x.note"]
NAMES=["dq/out/dq4_omop_20260825_r5/issueify_stmts/omop_issueify_0000.sql","dq/out/dq4_omop_20260825_r5/issueify_stmts/omop_issueify_0001.sql","dq/out/dq4_omop_20260825_r5/issueify_stmts/omop_issueify_0002.sql","dq/out/dq4_omop_20260825_r5/issueify_stmts/omop_issueify_0003.sql","dq/out/dq4_omop_20260825_r5/issueify_stmts/omop_issueify_0004.sql","dq/out/dq4_omop_20260825_r5/issueify_stmts/omop_issueify_0005.sql","dq/out/dq4_omop_20260825_r5/issueify_stmts/omop_issueify_0006.sql","dq/out/dq4_omop_20260825_r5/issueify_stmts/omop_issueify_0007.sql"]
TEMPLATE_REPLACEMENTS = {
    "dq4_omop_20260825_r5": RUN,
    "dq4_omop_20260825_r2": RUN,
    "2026-08-26T06:48:21.102Z": RUN_OPEN_TS,
    "cc682c9c-8795-4c48-adea-f988320f8d0d": SOURCE_UPDATE_ID,
    "f5c7c7ab-e37d-4a31-b9c2-b7631becb16a": SILVER_UPDATE_ID,
    "r5q9n3k6": SCRATCH_PREFIX,
}
def adapt(value):
    if isinstance(value, str):
        for old, new in TEMPLATE_REPLACEMENTS.items():
            value = value.replace(old, new)
        return value
    if isinstance(value, list):
        return [adapt(v) for v in value]
    if isinstance(value, dict):
        return {k: adapt(v) for k, v in value.items()}
    return value
STATEMENTS = adapt(STATEMENTS)
NAMES = adapt(NAMES)
def q(v):
    return "'" + str(v).replace("'", "''") + "'"
for seq,stmt in enumerate(STATEMENTS):
    sha=hashlib.sha256(stmt.encode()).hexdigest(); name=NAMES[seq]
    spark.sql(f"""INSERT INTO 8_dev.silver_qc.dq_exec_log (run_id,lane,stmt_file,seq,stmt_sha256,status,error_text,rows_affected,attempted_at,settled_at,created_by_session) VALUES ({q(RUN)},{q(LANE)},{q(name)},{seq},{q(sha)},'attempted',NULL,NULL,current_timestamp(),NULL,'DQ4')""")
    try:
        spark.sql(stmt)
        spark.sql(f"""INSERT INTO 8_dev.silver_qc.dq_exec_log (run_id,lane,stmt_file,seq,stmt_sha256,status,error_text,rows_affected,attempted_at,settled_at,created_by_session) VALUES ({q(RUN)},{q(LANE)},{q(name)},{seq},{q(sha)},'ok',NULL,NULL,current_timestamp(),current_timestamp(),'DQ4')""")
    except Exception as e:
        msg=str(e)[:4000]
        spark.sql(f"""INSERT INTO 8_dev.silver_qc.dq_exec_log (run_id,lane,stmt_file,seq,stmt_sha256,status,error_text,rows_affected,attempted_at,settled_at,created_by_session) VALUES ({q(RUN)},{q(LANE)},{q(name)},{seq},{q(sha)},'error',{q(msg)},NULL,current_timestamp(),current_timestamp(),'DQ4')""")
        raise